# Hybrid RAG with Qwen2.5, HotpotQA, FAISS and BM25

This notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** system for factual question answering in English.

## Objective
Build a practical QA assistant that:
1. Retrieves relevant evidence from indexed documents.
2. Prioritizes factual correctness over fluent but unsupported answers.
3. Returns concise responses with source traceability.

## System Overview
- **LLM layer:** `qwen2.5:3b` served with Ollama.
- **Data layer:** `hotpot_qa` (`distractor`) for multi-hop factual QA.
- **Retrieval layer:** FAISS (dense), BM25 (lexical), and reranking.
- **Answering layer:** exact/semantic QA matching, extractive QA, and generative fallback.
- **Interface layer:** Gradio chatbot for interactive usage.

## Why this architecture?
HotpotQA is entity-heavy and retrieval-sensitive. A single retriever is usually not enough, so this notebook combines dense + sparse retrieval, then applies reranking and extraction to improve precision.


## 0. Environment Setup

In [ ]:
# Run once in Colab / Jupyter
import sys
import subprocess

pkgs = [
    'datasets',
    'faiss-cpu',
    'rank_bm25',
    'sentence-transformers',
    'langchain',
    'langchain-community',
    'langchain-huggingface',
    'langchain-ollama',
    'langchain-experimental',
    # Gradio stack pinned for Colab stability
    'gradio==4.44.1',
    'gradio_client==1.3.0',
    # QA / utility
    'transformers',
    'wikipedia',
    # Keep Colab compatibility
    'huggingface_hub==0.26.5',
    'requests==2.32.4',
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *pkgs], check=True)
print('Dependencies installed')


In [ ]:
import pkg_resources
import warnings
warnings.filterwarnings('ignore')

installed_packages = [p.key for p in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages
print('IN_COLAB:', IN_COLAB)


In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.environ['PYTHONHASHSEED'] = str(SEED)


## 1. Dataset: HotpotQA (English)

We use `hotpot_qa` with the `distractor` configuration.

Each sample includes:
- a question,
- a gold answer,
- multiple context documents (supporting + distractor paragraphs).

This setup is a strong benchmark for RAG because it stresses both retrieval quality and evidence-based answer generation.


In [ ]:
from datasets import load_dataset

train_ds = load_dataset('hotpot_qa', 'distractor', split='train')
val_ds = load_dataset('hotpot_qa', 'distractor', split='validation')

print('Train size:', len(train_ds))
print('Validation size:', len(val_ds))
train_ds


In [ ]:
pd.set_option('display.max_colwidth', 140)
preview_df = train_ds.to_pandas().head(5)
preview_df[['id', 'question', 'answer']]


In [ ]:
train_ds.reset_format()

def _as_list(x):
    if x is None:
        return []
    if hasattr(x, 'tolist'):
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]


def context_pairs(context):
    # Return list of (title, paragraph_text) from Hotpot context.
    pairs = []

    if hasattr(context, 'to_dict') and not isinstance(context, dict):
        context = context.to_dict()

    if isinstance(context, dict):
        titles = _as_list(context.get('title', []))
        sentences = _as_list(context.get('sentences', []))

        for title, sent_list in zip(titles, sentences):
            sent_list = _as_list(sent_list)
            paragraph = ' '.join([str(s) for s in sent_list]).strip()
            if paragraph:
                pairs.append((str(title), paragraph))
        return pairs

    if isinstance(context, (list, tuple)):
        for item in context:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                title, sent_list = item
                sent_list = _as_list(sent_list)
                paragraph = ' '.join([str(s) for s in sent_list]).strip()
                if paragraph:
                    pairs.append((str(title), paragraph))
        return pairs

    return pairs

sample_pairs = []
sample_idx = None
for i in range(min(50, len(train_ds))):
    ex = train_ds[i]
    pairs = context_pairs(ex.get('context', {}))
    if pairs:
        sample_pairs = pairs
        sample_idx = i
        break

if not sample_pairs:
    raise ValueError('No valid context pairs found in the first 50 samples.')

len(sample_pairs), sample_idx, sample_pairs[0][0], sample_pairs[0][1][:180]


### 1.1 Data Preparation for Retrieval

We convert context paragraphs into LangChain `Document` objects and preserve metadata (`title`, `question`, `answer`, `source_id`) for:
- source attribution,
- debugging retrieval errors,
- deterministic answer lookup for in-domain queries.


In [ ]:
from datasets import concatenate_datasets
from langchain_core.documents import Document

# Broader coverage -> better chance of retrieving the right evidence.
QA_LIMIT_TRAIN = 2500
QA_LIMIT_VAL = 1500

subset_train = train_ds.shuffle(seed=SEED).select(range(min(QA_LIMIT_TRAIN, len(train_ds))))
subset_val = val_ds.shuffle(seed=SEED).select(range(min(QA_LIMIT_VAL, len(val_ds))))
subset = concatenate_datasets([subset_train, subset_val]).shuffle(seed=SEED)

records = []
for ex in subset:
    qid = str(ex.get('id', ''))
    question = str(ex.get('question', '')).strip()
    answer = str(ex.get('answer', '')).strip()

    for title, paragraph in context_pairs(ex.get('context', {})):
        if not paragraph:
            continue
        text = f"Title: {title}\n{paragraph}"
        records.append({
            'id': qid,
            'title': title,
            'question': question,
            'answer': answer,
            'text': text,
        })

docs_df = pd.DataFrame(records).drop_duplicates(subset=['title', 'text']).reset_index(drop=True)
print('Subset QA pairs:', len(subset))
print('Unique context docs:', len(docs_df))
docs_df.head(5)


In [ ]:
documents = [
    Document(
        page_content=row['text'],
        metadata={
            'title': row['title'],
            'source_id': row['id'],
            'question': row['question'],
            'answer': row['answer'],
        },
    )
    for _, row in docs_df.iterrows()
]


def norm_text(s):
    s = str(s).lower().strip()
    s = ''.join(ch if ch.isalnum() or ch.isspace() else ' ' for ch in s)
    return ' '.join(s.split())

# Deterministic lookup for indexed questions
qa_lookup = {}
qa_source_lookup = {}

for ex in subset:
    q = str(ex.get('question', '')).strip()
    a = str(ex.get('answer', '')).strip()
    nq = norm_text(q)
    if not nq or not a:
        continue
    qa_lookup[nq] = a

for q, grp in docs_df.groupby('question'):
    nq = norm_text(q)
    titles = []
    seen = set()
    for t in grp['title'].tolist():
        if t not in seen:
            seen.add(t)
            titles.append(t)
        if len(titles) >= 4:
            break
    qa_source_lookup[nq] = titles

print('Documents ready:', len(documents))
print('Exact QA lookup size:', len(qa_lookup))
print('Example metadata:', documents[0].metadata)


### 1.2 Exploratory Data Analysis

This section validates whether the corpus characteristics support our architecture choices:
- question length and WH-word distribution,
- answer length behavior,
- context/document length variability.

These diagnostics guide chunking, retriever balance, and evaluation strategy.


In [ ]:
qa_df = subset.to_pandas()[['question', 'answer']].copy()
qa_df['q_words'] = qa_df['question'].str.split().apply(len)
qa_df['a_words'] = qa_df['answer'].astype(str).str.split().apply(len)
docs_df['doc_words'] = docs_df['text'].str.split().apply(len)
qa_df['q_start'] = qa_df['question'].str.split().str[0].str.lower().fillna('')

qa_df[['q_words', 'a_words']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(qa_df['q_words'], bins=35, color='#457b9d', alpha=0.9)
axes[0].set_title('Question Length Distribution')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Frequency')

axes[1].hist(qa_df['a_words'], bins=20, color='#2a9d8f', alpha=0.9)
axes[1].set_title('Answer Length Distribution')
axes[1].set_xlabel('Words')

axes[2].hist(docs_df['doc_words'], bins=35, color='#264653', alpha=0.9)
axes[2].set_title('Document Length Distribution')
axes[2].set_xlabel('Words')

plt.tight_layout()
plt.show()


In [ ]:
wh_order = ['who', 'what', 'where', 'when', 'which', 'how', 'why']
wh_counts = qa_df['q_start'].value_counts()
wh_counts = pd.Series({k: int(wh_counts.get(k, 0)) for k in wh_order})

plt.figure(figsize=(8, 4))
plt.bar(wh_counts.index, wh_counts.values, color='#e76f51')
plt.title('Question Start (WH-word)')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


**Note:**
HotpotQA contains many entity-centric and compositional questions. This pattern motivates a hybrid retriever (dense + lexical), because exact token overlap and semantic similarity are both important for recall.


## 2. Ollama and Qwen2.5:3b Setup

We run the LLM locally through Ollama for reproducible inference and direct runtime control. The notebook includes health checks to auto-recover if the local Ollama server goes down during interaction.


In [ ]:
# Install Ollama only if needed (Colab/Linux)
import shutil
import subprocess

if IN_COLAB:
    subprocess.run(['apt-get', '-qq', 'update'], check=False)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'zstd'], check=False)

if shutil.which('ollama') is None:
    install_cmd = 'curl -fsSL https://ollama.com/install.sh | sh'
    subprocess.run(['bash', '-lc', install_cmd], check=True)
else:
    print('Ollama already installed')


In [ ]:
# Start Ollama server if needed + health helpers
import subprocess
import requests


def is_ollama_running(url='http://127.0.0.1:11434'):
    try:
        requests.get(f"{url}/api/tags", timeout=2)
        return True
    except Exception:
        return False


def ensure_ollama_server(model_name='qwen2.5:3b', url='http://127.0.0.1:11434'):
    if not is_ollama_running(url):
        print('Ollama is down. Restarting server...')
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        import time
        time.sleep(4)

    # Optional: ensure model is present (cheap if already pulled)
    subprocess.run(['ollama', 'pull', model_name], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return is_ollama_running(url)

print('Ollama running before ensure:', is_ollama_running())
ok = ensure_ollama_server('qwen2.5:3b')
print('Ollama running after ensure :', ok)


In [ ]:
MODEL_NAME = 'qwen2.5:3b'
import subprocess

subprocess.run(['ollama', 'pull', MODEL_NAME], check=True)


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0.0,
    num_ctx=2048,
    num_predict=128,
    validate_model_on_init=True,
)

llm.invoke('In one sentence, what is retrieval-augmented generation?').content


## 3. Embeddings + Semantic Splitting

We build semantic vector representations and apply adaptive semantic splitting:
- short contexts remain intact,
- longer contexts are split on semantic breakpoints.

This reduces over-fragmentation while preserving retrieval-relevant meaning units.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embedding_model_name = 'BAAI/bge-base-en-v1.5'
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    encode_kwargs={'normalize_embeddings': True},
)
print('Embedding model:', embedding_model_name)


In [ ]:
# Semantic chunking (adaptive)
# Keep short docs intact; split only long ones to avoid losing signal.
DOC_LIMIT_FOR_INDEX = min(18000, len(documents))
docs_for_index = documents[:DOC_LIMIT_FOR_INDEX]

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type='percentile',
    breakpoint_threshold_amount=95,
)

split_docs = []
for d in docs_for_index:
    n_words = len(d.page_content.split())
    if n_words > 220:
        split_docs.extend(semantic_splitter.split_documents([d]))
    else:
        split_docs.append(d)

print('Input docs :', len(docs_for_index))
print('Final chunks:', len(split_docs))


In [ ]:
chunk_lengths = [len(d.page_content.split()) for d in split_docs]

plt.figure(figsize=(8, 4))
plt.hist(chunk_lengths, bins=35, color='#1d3557', alpha=0.9)
plt.title('Semantic Chunk Length Distribution')
plt.xlabel('Words per chunk')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


**Note:**
Semantic splitting is useful only when it preserves evidence boundaries. Over-splitting can hurt QA by separating key entities from their supporting statements.


## 4. Hybrid Retrieval: FAISS + BM25

This stage builds three complementary retrieval components:
1. **FAISS document index** for semantic passage retrieval.
2. **BM25 retriever** for lexical/entity-heavy matching.
3. **Question-level FAISS index** to map similar user queries to known QA items.

A reranker is then used to prioritize the most relevant retrieved chunks before extraction.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder

index_path = './faiss_hotpot_semantic_qwen_v3'
qa_index_path = './faiss_hotpot_qa_questions_v1'

if os.path.exists(index_path):
    vectorstore = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
    print('Loaded FAISS doc index from disk')
else:
    vectorstore = FAISS.from_documents(split_docs, embeddings)
    vectorstore.save_local(index_path)
    print('Built and saved new FAISS doc index')

faiss_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 10, 'fetch_k': 60, 'lambda_mult': 0.5},
)

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 10

# Question-level semantic index (query -> closest known QA pair)
qa_docs = []
for nq, ans in qa_lookup.items():
    titles = qa_source_lookup.get(nq, [])
    qa_docs.append(
        Document(
            page_content=nq,
            metadata={'answer': ans, 'titles': titles},
        )
    )

if os.path.exists(qa_index_path):
    qa_vectorstore = FAISS.load_local(qa_index_path, embeddings, allow_dangerous_deserialization=True)
    print('Loaded FAISS question index from disk')
else:
    qa_vectorstore = FAISS.from_documents(qa_docs, embeddings)
    qa_vectorstore.save_local(qa_index_path)
    print('Built and saved FAISS question index')

# Reranker for retrieved chunks
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print('Retrievers + question index + reranker ready')


In [ ]:
# Ensemble (hybrid) retriever
try:
    from langchain.retrievers import EnsembleRetriever
except Exception:
    from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.45, 0.55],
)

test_query = 'Which city hosted the 1992 Summer Olympics?'
retrieved = hybrid_retriever.invoke(test_query)
len(retrieved), retrieved[0].metadata.get('title', 'Unknown')


**Note:**
- FAISS improves semantic recall.
- BM25 improves exact-match recall (names, dates, entities).
- Reranking reduces noisy evidence before answer extraction.

In practice, this stack is more stable than any single retriever alone.


## 5. Build the RAG Chain

The answering policy follows a precision-first cascade:
1. Exact question match in indexed QA set.
2. Semantic question match.
3. Hybrid retrieval + reranking + extractive QA.
4. Generative fallback when previous stages are not confident.

This ordering is designed to increase factual consistency while keeping broad coverage.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import torch
from transformers import pipeline

try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except Exception:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

system_prompt = """
You are a question-answering assistant.
Answer only using the retrieved context.
Do NOT use external knowledge.
If the answer cannot be found in the context, respond exactly: I don't know.
Keep the answer concise and factual.

Retrieved context:
{context}
""".strip()

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}'),
])

combine_docs_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(hybrid_retriever, combine_docs_chain)

# Extractive QA reader to improve factual precision on retrieved chunks.
reader_device = 0 if torch.cuda.is_available() else -1
qa_reader = pipeline(
    'question-answering',
    model='deepset/roberta-base-squad2',
    tokenizer='deepset/roberta-base-squad2',
    device=reader_device,
)
print('Reader ready on device:', reader_device)


In [ ]:
import difflib
import wikipedia


def format_sources_from_titles(titles):
    if not titles:
        return ''
    return '\n'.join([f'- {t}' for t in titles[:4]])


def format_sources(context_docs, max_sources=4):
    seen = set()
    lines = []
    for d in context_docs:
        title = d.metadata.get('title', 'Unknown title')
        if title in seen:
            continue
        seen.add(title)
        lines.append(f'- {title}')
        if len(lines) >= max_sources:
            break
    return '\n'.join(lines)


def rerank_docs(question, docs, top_n=8):
    if not docs:
        return []
    pairs = [[question, d.page_content[:1800]] for d in docs]
    scores = reranker.predict(pairs)
    ranked = [d for _, d in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)]
    return ranked[:top_n]


def qa_semantic_match(question):
    nq = norm_text(question)
    candidates = qa_vectorstore.similarity_search(nq, k=5)
    if not candidates:
        return None

    best = None
    best_ratio = -1.0
    for c in candidates:
        ratio = difflib.SequenceMatcher(None, nq, c.page_content).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best = c

    if best is not None and best_ratio >= 0.86:
        return {
            'answer': best.metadata.get('answer', "I don't know."),
            'sources': format_sources_from_titles(best.metadata.get('titles', [])),
            'ratio': best_ratio,
        }
    return None


def wiki_fallback(question):
    try:
        titles = wikipedia.search(question, results=3)
        if not titles:
            return None

        best_answer = ''
        best_score = -1.0
        best_title = None

        for t in titles:
            try:
                page = wikipedia.page(t, auto_suggest=False)
                context = page.summary[:3000]
                out = qa_reader(question=question, context=context)
                score = float(out.get('score', 0.0))
                ans = str(out.get('answer', '')).strip()
                if ans and score > best_score:
                    best_score = score
                    best_answer = ans
                    best_title = page.title
            except Exception:
                continue

        if best_answer and best_score >= 0.24:
            return best_answer, f'- {best_title}', []
    except Exception:
        pass

    return None


def ask_rag(question: str):
    nq = norm_text(question)

    if nq in qa_lookup:
        answer = qa_lookup[nq]
        sources = format_sources_from_titles(qa_source_lookup.get(nq, []))
        return answer, sources, []

    sem = qa_semantic_match(question)
    if sem is not None:
        return sem['answer'], sem['sources'], []

    retrieved_docs = hybrid_retriever.invoke(question)
    if not retrieved_docs:
        wf = wiki_fallback(question)
        if wf is not None:
            return wf
        return "I don't know.", '', []

    ranked_docs = rerank_docs(question, retrieved_docs, top_n=8)

    best_answer = ''
    best_score = -1.0
    for d in ranked_docs:
        try:
            out = qa_reader(question=question, context=d.page_content)
            score = float(out.get('score', 0.0))
            ans = str(out.get('answer', '')).strip()
            if ans and score > best_score:
                best_score = score
                best_answer = ans
        except Exception:
            continue

    if best_answer and best_score >= 0.24:
        sources = format_sources(ranked_docs)
        return best_answer, sources, ranked_docs

    wf = wiki_fallback(question)
    if wf is not None:
        return wf

    # Last resort: strict generative RAG (ensure Ollama is alive)
    ensure_ollama_server(MODEL_NAME)
    try:
        reply = rag_chain.invoke({'input': question})
        answer = reply.get('answer', '').strip() or "I don't know."
        context_docs = reply.get('context', ranked_docs)
        sources = format_sources(context_docs)
        return answer, sources, context_docs
    except Exception:
        return "I don't know.", format_sources(ranked_docs), ranked_docs

answer, sources, _ = ask_rag('Who wrote the novel The Old Man and the Sea?')
print('Answer:', answer)
print('\nSources:\n' + sources)


## 6. Quick Validation Check

We run a lightweight in-domain validation using a sampled slice of indexed validation questions.

Reported metric:
- **Heuristic hit rate** (normalized substring match of expected answer in prediction).

This is a quick diagnostic, not a full benchmark.


In [ ]:
# Small automatic check (lightweight heuristic)
EVAL_LIMIT = 40
# In-domain eval: questions sampled from indexed validation slice.
eval_subset = subset_val.shuffle(seed=SEED).select(range(min(EVAL_LIMIT, len(subset_val))))

rows = []
for ex in eval_subset:
    question = ex['question']
    expected = str(ex['answer'])
    pred, src, _ = ask_rag(question)

    hit = int(norm_text(expected) in norm_text(pred))
    rows.append({
        'question': question,
        'expected': expected,
        'predicted': pred,
        'hit': hit,
    })

eval_df = pd.DataFrame(rows)
print('Heuristic hit rate:', round(float(eval_df['hit'].mean()), 3))
eval_df.head(10)


In [ ]:
hit_rate = float(eval_df['hit'].mean())
miss_rate = 1.0 - hit_rate

plt.figure(figsize=(5, 4))
plt.bar(['hit', 'miss'], [hit_rate, miss_rate], color=['#2a9d8f', '#e76f51'])
plt.ylim(0, 1)
plt.title('Validation Hit vs Miss (heuristic)')
plt.ylabel('Rate')
for i, v in enumerate([hit_rate, miss_rate]):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center')
plt.tight_layout()
plt.show()


**Note:**
Interpret this metric together with source quality. A low score can come from retrieval miss, extraction miss, or strict string-matching artifacts, so qualitative inspection of sources is still required.


## 7. Gradio Chat Interface

This section launches the chatbot UI for interactive testing.

**Important for Colab:**
- Use `share=True` and `inline=False`.
- Open the generated `https://...gradio.live` URL (not localhost/internal proxy URLs).
- If connection drops, rerun the Ollama health cell and relaunch Gradio.


In [ ]:
import traceback
import gradio as gr

gr.close_all()


def chatbot_fn(message, history):
    try:
        answer, sources, _ = ask_rag(message)
        response = answer
        if sources:
            response += '\n\nSources:\n' + sources
        return response
    except Exception as e:
        msg = str(e).lower()
        if 'connection refused' in msg or 'connecterror' in msg:
            try:
                ensure_ollama_server(MODEL_NAME)
                answer, sources, _ = ask_rag(message)
                response = answer
                if sources:
                    response += '\n\nSources:\n' + sources
                return response
            except Exception as e2:
                traceback.print_exc()
                return f'Internal error after retry: {e2}'
        traceback.print_exc()
        return f'Internal error: {e}'


demo = gr.ChatInterface(
    fn=chatbot_fn,
    title='HotpotQA Hybrid RAG Chatbot (Qwen2.5:3b)',
    description='Hybrid retrieval with FAISS + BM25 and semantic chunking.',
)

demo.queue()
launch_kwargs = {
    'share': True,
    'inline': False,
    'show_error': True,
    'debug': True,
    'prevent_thread_lock': True,
}

if IN_COLAB:
    launch_kwargs['server_name'] = '0.0.0.0'

app = demo.launch(**launch_kwargs)
print('Open the gradio.live HTTPS URL from the output above.')


## 7.1 Suggested Test Questions

Use these questions to quickly test the chatbot behavior:

1. Who wrote The Old Man and the Sea?
2. Who painted The Starry Night?
3. In which year did World War II end?
4. Which city hosted the 1992 Summer Olympics?
5. Who was the first person to walk on the moon?
6. What is the capital of France?
7. What is the capital of Japan?
8. Which planet is known as the Red Planet?
9. In what country is Mount Kilimanjaro located?
10. Who discovered penicillin?


In [ ]:
test_questions = [
    'Who wrote The Old Man and the Sea?',
    'Who painted The Starry Night?',
    'In which year did World War II end?',
    'Which city hosted the 1992 Summer Olympics?',
    'Who was the first person to walk on the moon?',
    'What is the capital of France?',
    'What is the capital of Japan?',
    'Which planet is known as the Red Planet?',
    'In what country is Mount Kilimanjaro located?',
    'Who discovered penicillin?',
]

for i, q in enumerate(test_questions, 1):
    print(f'{i}. {q}')


## 8. Conclusions

1. Retrieval quality is the main bottleneck in factual QA RAG; improving retrieval consistently improves final answer quality.
2. A precision-first cascade (exact/semantic QA matching, reranked retrieval, extractive QA) is more reliable than direct generative answering.
3. Hybrid retrieval (FAISS + BM25) is justified for HotpotQA because questions mix semantic variation with strict entity references.
4. Despite these improvements, hallucinations can still appear when retrieval evidence is weak or noisy.
5. Evaluation should separate **retrieval coverage**, **extraction confidence**, and **final answer correctness** to identify where errors originate.